# ECOS 무위험(CD91) · KRX 폐지 이력 — P0 데이터 가용성 프로브

**목적:** P0의 남은 데이터 결정 두 개를 닫는다.
1. **ECOS CD91 무위험** — 91일 CD 금리의 시작 시점·단위 확인 (5b 초과수익 RF용).
2. **KRX 폐지 이력** — 폐지 종목·폐지일의 백커버리지·완전성 확인 (D10 생존편향 로깅 + 패널 완전성용).

D8 = **KOSPI+KOSDAQ 전체시장** 확정 → 두 시장의 폐지 이력이 모두 필요하다.

**설치:** `pip install PublicDataReader finance-datareader`

**ECOS 키:** ECOS는 별도 인증키가 필요하다(OpenDART·KRX와 다른 시스템). 없으면 Section 2는 건너뛰고, 발급 후 재실행.
- ecos.bok.or.kr/api → 인증키 신청 → 본인인증 → 이메일/정보 입력 → 인증키 이메일 수령(즉시).
- 받은 키를 프로젝트 루트 `.env`에 `ECOS_API_KEY=...` 한 줄 추가 후 커널 재시작.

**참고:** Section 1(폐지 이력)은 FinanceDataReader라 키가 필요 없다 — 지금 바로 실행 가능.

In [2]:
# [0] env + imports
import os
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import FinanceDataReader as fdr

# ECOS는 키 있을 때만
ecos = None
try:
    from PublicDataReader import Ecos
    ECOS_KEY = os.environ.get("ECOS_API_KEY")
    if ECOS_KEY:
        ecos = Ecos(ECOS_KEY)
        print("ECOS: 준비 완료")
    else:
        print("ECOS: ECOS_API_KEY 없음 — Section 2 건너뜀(키 발급 후 .env 추가 + 커널 재시작)")
except ImportError:
    print("ECOS: PublicDataReader 미설치 — pip install PublicDataReader")

print("FinanceDataReader:", getattr(fdr, "__version__", "?"))

def pick_col(df, keys):
    """컬럼명에 keys 중 하나라도 들어간 첫 컬럼 반환(대소문자 무시)."""
    for c in df.columns:
        cu = str(c).upper()
        if any(k.upper() in cu for k in keys):
            return c
    return None

ECOS: 준비 완료
FinanceDataReader: 0.9.202


## Section 1 — KRX 폐지 이력 (FinanceDataReader · 키 불필요)

`StockListing('KRX-DELISTING')`은 KRX 상장폐지 전체 종목을 반환한다(1956년대까지 소급). D10 생존편향 로깅과 패널 완전성의 근거.

In [3]:
# [1] 폐지 종목 전체 + 스키마 확인
delisted = fdr.StockListing("KRX-DELISTING")
print("폐지 종목 총수:", len(delisted))
print("컬럼:", list(delisted.columns))
delisted.head()

폐지 종목 총수: 4146
컬럼: ['Symbol', 'Name', 'Market', 'SecuGroup', 'Kind', 'ListingDate', 'DelistingDate', 'Reason', 'ArrantEnforceDate', 'ArrantEndDate', 'Industry', 'ParValue', 'ListingShares', 'ToSymbol', 'ToName']


,Symbol,Name,Market,SecuGroup,Kind,ListingDate,DelistingDate,Reason,ArrantEnforceDate,ArrantEndDate,Industry,ParValue,ListingShares,ToSymbol,ToName
0,028740,경성전기,KOSPI,주권,NaN,1956-03-03,1961-06-30,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
1,028730,남선전기,KOSPI,주권,NaN,1956-03-03,1961-06-30,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
2,034380,조선맥주,KOSPI,주권,NaN,1956-10-01,1960-11-26,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
3,028720,수도극장,KOSPI,주권,NaN,1957-07-01,1960-11-21,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
4,028750,한국운수,KOSPI,주권,NaN,1956-03-03,1962-01-04,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN


In [4]:
# [2] 폐지 이력 커버리지 분석
d = delisted.copy()
dcol = pick_col(d, ["DelistingDate", "폐지"])
lcol = pick_col(d, ["ListingDate", "상장일"])
mcol = pick_col(d, ["Market", "시장"])
scol = pick_col(d, ["SecuGroup", "Kind", "증권"])

d[dcol] = pd.to_datetime(d[dcol], errors="coerce")
print("폐지일 컬럼:", dcol)
print("폐지일 범위:", d[dcol].min(), "~", d[dcol].max())
print("폐지일 결측:", int(d[dcol].isna().sum()), "/", len(d))

if mcol:
    print()
    print("시장별 폐지 종목수:")
    print(d[mcol].value_counts())

if scol:
    print()
    print(f"{scol}별(상위 10):")
    print(d[scol].value_counts().head(10))

print()
print("폐지 10년단위 추이(백커버리지 확인):")
d["decade"] = (d[dcol].dt.year // 10) * 10
print(d.groupby("decade").size())

폐지일 컬럼: DelistingDate
폐지일 범위: 1960-11-21 00:00:00 ~ 2026-06-30 00:00:00
폐지일 결측: 0 / 4146

시장별 폐지 종목수:
Market
KOSPI     2124
KOSDAQ    1903
KONEX      119
Name: count, dtype: int64

SecuGroup별(상위 10):
SecuGroup
주권         2086
신주인수권증서     854
수익증권        779
투자회사        176
신주인수권증권     160
선박투자회사       55
부동산투자회사      18
외국주권         12
주식예탁증권        6
Name: count, dtype: int64

폐지 10년단위 추이(백커버리지 확인):
decade
1960       6
1970      35
1980      79
1990     476
2000    1334
2010    1200
2020    1016
dtype: int64


## 패널 완전성 — 현재상장 ∪ 폐지 = 생존편향 없는 전체 유니버스

FDR의 `KRX-DESC`는 **현재 상장만** 반환한다. 생존편향 없는 패널은 (현재상장 ∪ 폐지)의 합집합이어야 한다. D8이 KOSPI+KOSDAQ이므로 두 시장 모두 포함.

In [5]:
# [3] 현재상장 규모 + all-time 유니버스 규모 추정
listed = fdr.StockListing("KRX-DESC")
print("현재 상장:", len(listed))
print("컬럼:", list(listed.columns))
lmcol = pick_col(listed, ["Market", "시장"])
if lmcol:
    print()
    print("현재상장 시장별:")
    print(listed[lmcol].value_counts())
print()
print(f"All-time 유니버스 규모 ≈ {len(listed)}(현재) + {len(delisted)}(폐지) = {len(listed)+len(delisted)}")
print("→ 생존편향 없는 패널 = 이 합집합. v0.1은 KOSPI+KOSDAQ 보통주로 필터(우선주·ETF·리츠 제외).")

현재 상장: 2876
컬럼: ['Code', 'Name', 'Market', 'Sector', 'Industry', 'Products', 'ListingDate', 'SettleMonth', 'Representative', 'HomePage', 'Region']

현재상장 시장별:
Market
KOSDAQ           1774
KOSPI             945
KONEX             107
KOSDAQ GLOBAL      50
Name: count, dtype: int64

All-time 유니버스 규모 ≈ 2876(현재) + 4146(폐지) = 7022
→ 생존편향 없는 패널 = 이 합집합. v0.1은 KOSPI+KOSDAQ 보통주로 필터(우선주·ETF·리츠 제외).


## Section 2 — ECOS CD91 무위험 (PublicDataReader · 키 필요)

통계표 코드와 CD(91일) 항목코드·가용기간을 **자동 발견**한다(하드코딩 최소화). 시장금리(일별) 표는 보통 `817Y002`.

In [8]:
# [4+5] CD91 항목코드 + 수록기간 발견 (컬럼 자동적응 + 진단)
STAT = "817Y002"   # 시장금리(일별). 만약 이 줄에서 에러나면 그 메시지를 알려줘.

def col_like(df, *keys):
    for c in df.columns:
        s = str(c).upper()
        if any(k.upper() in s for k in keys):
            return c
    return None

items = ecos.get_statistic_item_list(통계표코드=STAT)
print("=== item_list 컬럼 ===")
print(list(items.columns), "| 행수:", len(items))
print()

namecol  = col_like(items, "항목명", "ITEM_NAME", "CONT_NAME", "명", "NAME")
codecol  = col_like(items, "항목코드", "ITEM_CODE", "CONT_CODE", "코드", "CODE")
startcol = col_like(items, "수록시작", "시작", "START")
endcol   = col_like(items, "수록종료", "종료", "END")
cyclecol = col_like(items, "주기", "CYCLE")
print("탐지: name=", namecol, "| code=", codecol, "| start=", startcol, "| end=", endcol)
print()

CD91_CODE = None
if namecol is None or codecol is None:
    print("항목명/코드 자동탐지 실패. 아래 전체 보고 CD91 행 확인(컬럼명도 알려줘):")
    print(items.head(30).to_string())
else:
    cd = items[items[namecol].astype(str).str.contains("CD", na=False)]
    show = [c for c in [codecol, namecol, cyclecol, startcol, endcol] if c]
    print("=== CD 관련 항목 ===")
    print(cd[show].to_string(index=False) if len(cd) else "(CD 항목 없음)")
    cd91 = cd[cd[namecol].astype(str).str.contains("91", na=False)] if len(cd) else cd
    src = cd91 if len(cd91) else cd
    if len(src):
        CD91_CODE = str(src.iloc[0][codecol])
    print()
    print(">>> CD91_CODE =", CD91_CODE)
    if startcol and len(src):
        print(">>> CD91 수록기간:", src.iloc[0][startcol], "~", (src.iloc[0][endcol] if endcol else "?"))

=== item_list 컬럼 ===
['통계표코드', '통계명', '항목그룹코드', '항목그룹명', '통계항목코드', '통계항목명', '상위통계항목코드', '상위통계항목명', '시점', '수록시작일자', '수록종료일자', '자료수', '단위', '가중치'] | 행수: 27

탐지: name= 통계명 | code= 통계표코드 | start= 수록시작일자 | end= 수록종료일자

=== CD 관련 항목 ===
(CD 항목 없음)

>>> CD91_CODE = None


In [9]:
# [6] CD91 표본 조회 — 단위·실제 시작 확인 (보조 확인용)
s = ecos.get_statistic_search(
    통계표코드=STAT, 주기="D",
    검색시작일자="19900101", 검색종료일자="19951231",
    통계항목코드1=CD91_CODE)
print("=== search 컬럼 ===")
print(list(s.columns) if s is not None else "None 반환", "| 행수:", 0 if s is None else len(s))
print()
if s is not None and len(s):
    print(s.head(8).to_string(index=False))
    tcol = col_like(s, "시점", "TIME")
    ucol = col_like(s, "단위", "UNIT")
    if tcol: print("이 구간 최초 시점:", s[tcol].min())
    if ucol: print("단위:", s[ucol].iloc[0])
else:
    print("1990~95 데이터 없음 → 검색시작일자를 2000으로 올려 재시도(또는 위 수록시작 참고).")
print()
print("RF 변환(5b): CD91 연율 % → 월별 = (1+r/100)**(1/12)-1  또는  r/100/12.")

=== search 컬럼 ===
['통계표코드', '통계명', '통계항목코드1', '통계항목명1', '통계항목코드2', '통계항목명2', '통계항목코드3', '통계항목명3', '통계항목코드4', '통계항목명4', '단위', 'WGT', '시점', '값'] | 행수: 1441

  통계표코드               통계명   통계항목코드1        통계항목명1 통계항목코드2 통계항목명2 통계항목코드3 통계항목명3 통계항목코드4 통계항목명4 단위  WGT       시점     값
817Y002 1.3.2.1. 시장금리(일별) 010101000 콜금리(1일, 전체거래)    None   None    None   None    None   None 연% None 19950103 13.06
817Y002 1.3.2.1. 시장금리(일별) 010300000  회사채(3년, AA-)    None   None    None   None    None   None 연% None 19950103 14.25
817Y002 1.3.2.1. 시장금리(일별) 010400001      통안증권(1년)    None   None    None   None    None   None 연% None 19950103 13.85
817Y002 1.3.2.1. 시장금리(일별) 010502000       CD(91일)    None   None    None   None    None   None 연% None 19950103  14.7
817Y002 1.3.2.1. 시장금리(일별) 010503000       CP(91일)    None   None    None   None    None   None 연% None 19950103 15.37
817Y002 1.3.2.1. 시장금리(일별) 010101000 콜금리(1일, 전체거래)    None   None    None   None    None   None 연% None 19950104 11.63
817Y002 1.3.2.1. 시장

## Section 3 — 요약 판정

이 프로브가 닫는 두 결정:

- **RF 시작 시점(ECOS CD91):** Section 2의 가용기간 시작이 무위험 표본 바닥. PBR 횡단면 2003·시가총액 1995와 비교해 **무엇이 표본 바닥을 묶는 제약인지** 확인. (CD91이 1990년대부터면 RF는 제약이 아님 → 바닥은 PBR 2003.)
- **폐지 이력 백커버리지(KRX-DELISTING):** Section 1의 폐지일 범위·시장별 분포가 D10 생존편향 로깅 가능 범위. 폐지일 결측이 적고 KOSPI·KOSDAQ 둘 다 잡히면 → 생존편향 없는 패널 구성 가능.

**다음:** 두 시작 시점을 PRD §0/§2에 기입 → P0 완전히 종료 → 5a ETL 설계 착수.
표본 바닥 후보 정리: MKT·SMB = 시총 1995, **HML = PBR 2003(유력 바인딩)**, RF = (이 프로브로 확정).